# Jazz Knowledge Graph — Exploration
End-to-end exploration of the Jazz KG built from Wikipedia + Wikidata.

## 1. Setup

In [4]:
import os
import sys
from pathlib import Path

# Navigate to project root
PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import rdflib
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, OWL
import matplotlib.pyplot as plt
import matplotlib
import networkx as nx
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

WDT = Namespace('http://www.wikidata.org/prop/direct/')
WD  = Namespace('http://www.wikidata.org/entity/')
JAZZ = Namespace('http://jazz-kg.org/ontology/')

EXPANDED_NT = PROJECT_ROOT / 'kg_artifacts' / 'expanded.nt'
INITIAL_TTL = PROJECT_ROOT / 'kg_artifacts' / 'initial_kg.ttl'

g = Graph()
if EXPANDED_NT.exists():
    g.parse(str(EXPANDED_NT), format='nt')
    print(f'Loaded expanded.nt: {len(g):,} triples')
elif INITIAL_TTL.exists():
    g.parse(str(INITIAL_TTL), format='turtle')
    print(f'Loaded initial_kg.ttl: {len(g):,} triples')
else:
    print('WARNING: No KG artifact found. Run the pipeline first.')

AttributeError: 'wrapper_descriptor' object has no attribute '__annotate__'

## 2. Global Stats

In [ ]:
subjects   = set(g.subjects())
predicates = set(g.predicates())
objects    = set(g.objects())

print(f'Total triples    : {len(g):,}')
print(f'Unique subjects  : {len(subjects):,}')
print(f'Unique predicates: {len(predicates):,}')
print(f'Unique objects   : {len(objects):,}')

# Count types
type_counts = Counter()
for _, _, o in g.triples((None, RDF.type, None)):
    type_counts[str(o).split('/')[-1]] += 1

print('\nTop types:')
for t, c in type_counts.most_common(10):
    print(f'  {t}: {c}')

## 3. Top Entities — Musicians by Triple Count

In [ ]:
subject_counts = Counter(str(s) for s in g.subjects())

# Filter to jazz-kg entities (not Wikidata)
jazz_entities = {k: v for k, v in subject_counts.items() if 'jazz-kg.org' in k}
top20 = sorted(jazz_entities.items(), key=lambda x: x[1], reverse=True)[:20]

if top20:
    labels = [u.split('/')[-1].replace('_', ' ')[:30] for u, _ in top20]
    counts = [c for _, c in top20]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(labels[::-1], counts[::-1], color='steelblue')
    ax.set_xlabel('Number of triples')
    ax.set_title('Top 20 Jazz KG Entities by Triple Count')
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'reports' / 'top_entities.png', dpi=120)
    plt.show()
else:
    print('No jazz-kg.org entities found — check pipeline outputs.')

## 4. Relation Graph — Miles Davis Neighbourhood

In [ ]:
# Find Miles Davis node
miles_node = None
for s in g.subjects():
    if 'miles_davis' in str(s).lower() or 'Miles_Davis' in str(s):
        miles_node = s
        break

if miles_node:
    G = nx.DiGraph()
    center = str(miles_node).split('/')[-1]
    G.add_node(center)

    neighbours = list(g.triples((miles_node, None, None)))[:20]
    for s, p, o in neighbours:
        pred = str(p).split('/')[-1]
        obj  = str(o).split('/')[-1][:25]
        G.add_edge(center, obj, label=pred)

    fig, ax = plt.subplots(figsize=(12, 8))
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_color='lightblue',
            node_size=1200, font_size=8, ax=ax, arrows=True)
    ax.set_title('Miles Davis — 1-hop Neighbourhood (up to 20 edges)')
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'reports' / 'miles_davis_graph.png', dpi=120)
    plt.show()
else:
    print('Miles Davis node not found in KG.')

## 5. Entity Type Distribution

In [ ]:
import pandas as pd

csv_path = PROJECT_ROOT / 'data' / 'extracted_knowledge.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    type_dist = df['entity_type'].value_counts()

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.pie(type_dist.values, labels=type_dist.index, autopct='%1.1f%%', startangle=140)
    ax.set_title('Entity Type Distribution')
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'reports' / 'entity_types.png', dpi=120)
    plt.show()
    print(type_dist)
else:
    print('extracted_knowledge.csv not found.')

## 6. Timeline — Musician Birth Years (wdt:P569)

In [ ]:
import re

birth_years = []
P569 = WDT.P569
for _, _, o in g.triples((None, P569, None)):
    m = re.search(r'(\d{4})', str(o))
    if m:
        year = int(m.group(1))
        if 1850 <= year <= 2010:
            birth_years.append(year)

if birth_years:
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.hist(birth_years, bins=30, color='coral', edgecolor='white')
    ax.set_xlabel('Birth Year')
    ax.set_ylabel('Count')
    ax.set_title('Distribution of Musician Birth Years (wdt:P569)')
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / 'reports' / 'birth_years.png', dpi=120)
    plt.show()
    print(f'{len(birth_years)} birth years found')
else:
    print('No P569 (birth date) triples found in expanded KG.')

## 7. Interactive Graph — PyVis

In [ ]:
try:
    from pyvis.network import Network

    net = Network(height='600px', width='100%', bgcolor='#222222', font_color='white',
                  directed=True, notebook=True)
    net.barnes_hut()

    # Take up to 80 jazz-kg triples for the interactive view
    jazz_triples = [(s, p, o) for s, p, o in g
                    if 'jazz-kg.org' in str(s)][:80]

    added_nodes = set()
    for s, p, o in jazz_triples:
        sn = str(s).split('/')[-1][:30]
        on = str(o).split('/')[-1][:30]
        pn = str(p).split('/')[-1][:20]
        if sn not in added_nodes:
            net.add_node(sn, label=sn, color='#4FC3F7')
            added_nodes.add(sn)
        if on not in added_nodes:
            net.add_node(on, label=on, color='#FFB74D')
            added_nodes.add(on)
        net.add_edge(sn, on, title=pn)

    reports_dir = PROJECT_ROOT / 'reports'
    reports_dir.mkdir(exist_ok=True)
    html_path = reports_dir / 'kg_graph.html'
    net.save_graph(str(html_path))
    print(f'Interactive graph saved to {html_path}')
    net.show(str(html_path))
except ImportError:
    print('pyvis not installed. Run: pip install pyvis')

## 8. Top Albums per Label

In [ ]:
if csv_path.exists():
    df = pd.read_csv(csv_path)
    # Filter relations involving record labels and albums
    label_albums = df[
        (df['relation_type'].str.contains('label|release|publish', case=False, na=False))
        | (df['entity_type'].isin(['ORG']))
    ].copy()

    if not label_albums.empty:
        pivot = label_albums.groupby(['relation_to', 'entity']).size().reset_index(name='count')
        pivot = pivot[pivot['relation_to'] != ''].nlargest(20, 'count')
        print(pivot.to_string(index=False))
    else:
        print('No label→album relations found in extracted_knowledge.csv')
else:
    print('extracted_knowledge.csv not found.')

## 9. SPARQL Query Demo

In [ ]:
# Example SPARQL query on the local KG
SPARQL_QUERY = """
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX jazz: <http://jazz-kg.org/ontology/>

SELECT ?musician ?label
WHERE {
  ?musician rdf:type jazz:Musician .
  OPTIONAL { ?musician rdfs:label ?label . }
}
LIMIT 20
"""

results = list(g.query(SPARQL_QUERY))
if results:
    print(f'SPARQL returned {len(results)} results:')
    for row in results:
        musician = str(row.musician).split('/')[-1]
        label = str(row.label) if row.label else ''
        print(f'  {musician}  {label}')
else:
    print('No results. Check that jazz:Musician class is used in the KG.')

## 10. Knowledge Graph Embeddings

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

metrics_path = PROJECT_ROOT / "reports" / "kge_metrics.json"
tsne_path = PROJECT_ROOT / "reports" / "tsne_embeddings.png"

if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    print("KGE Evaluation Metrics:")
    for model, scores in metrics.items():
        print(f"\n{model}:")
        for k, v in scores.items():
            print(f"  {k}: {v:.4f}")
else:
    print("kge_metrics.json not found — run src/kge/run_kge.py first")

if tsne_path.exists():
    img = mpimg.imread(str(tsne_path))
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title("Entity Embeddings — PCA/t-SNE projection")
    plt.tight_layout()
    plt.show()
else:
    print("tsne_embeddings.png not found")

## 11. OWL Reasoning

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from reason.reasoner import Reasoner

reasoner = Reasoner(g)
n_inferred = reasoner.infer_new_facts()
consistent = reasoner.validate_consistency()
print(f"New facts inferred: {n_inferred}")
print(f"Graph consistent: {consistent}")
print(f"Total triples after reasoning: {len(g)}")